<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW12 · Session 1 · Bar Charts, Line Charts, and Histograms

        **Course level:** developing beginner  
        **Prior learning:** clean pandas DataFrames, grouping, dates, and descriptive statistics  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        A useful chart begins with a question, selects an appropriate visual encoding, and removes anything that distracts from the evidence.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - match bar, line, and histogram charts to suitable questions
- create labelled figures with matplotlib
- aggregate data before drawing a category comparison
- plot time in chronological order
- choose and explain histogram bins


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load plotting tools and datasets

We use matplotlib directly so every chart element is visible and controllable.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

sales_raw = pd.read_csv(DATA_DIR / "business_sales_dirty.csv")
monthly = pd.read_csv(DATA_DIR / "monthly_sales.csv", parse_dates=["month"])
print(sales_raw.shape, monthly.shape)


## 1. Choose a chart from the question

| Question | Strong starting chart |
|---|---|
| How do categories compare? | bar chart |
| How does a measure change through time? | line chart |
| What is the shape of one numerical distribution? | histogram |

The same dataset can support many charts, but not every chart answers the same question.


### Prepare a clean plotting table

Charts should not silently hide invalid values. We standardise text, parse dates, remove exact duplicates, and calculate net revenue.


In [ ]:
sales = sales_raw.copy()
sales["region"] = sales["region"].astype("string").str.strip().str.title().fillna("Unknown")
sales["channel"] = sales["channel"].astype("string").str.strip().str.title().replace({"Telephone": "Phone"})
sales["order_date"] = pd.to_datetime(
    sales["order_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)
sales["units"] = sales["units"].mask(~sales["units"].between(1, 20))
sales["unit_price"] = sales["unit_price"].mask(sales["unit_price"] <= 0)
sales = sales.drop_duplicates().dropna(
    subset=["order_date", "units", "unit_price"]
).copy()
sales["net_revenue"] = sales["units"] * sales["unit_price"] * (1 - sales["discount"])
print("Plotting rows:", len(sales))


## 2. Bar charts compare categories

Aggregate first, sort intentionally, label units, and begin a zero-based numerical axis unless there is a strong reason not to. A bar length encodes magnitude, so a truncated axis exaggerates differences.


### Plot revenue by region

The data preparation and plotting are separate, making both easier to check.


In [ ]:
revenue_by_region = (
    sales.groupby("region")["net_revenue"]
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(7, 4))
revenue_by_region.plot(kind="bar", ax=ax, color="#2563eb")
ax.set_title("Net revenue by region")
ax.set_xlabel("Region")
ax.set_ylabel("Net revenue (£)")
ax.tick_params(axis="x", rotation=0)
ax.set_ylim(bottom=0)
fig.tight_layout()
plt.show()
plt.close(fig)


### Add direct value labels

Labels reduce lookup effort when exact totals matter. Round at display time, not during calculation.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(revenue_by_region.index, revenue_by_region.values, color="#0f766e")
ax.bar_label(bars, labels=[f"£{value:,.0f}" for value in revenue_by_region.values], padding=3)
ax.set_title("Net revenue by region")
ax.set_ylabel("Net revenue (£)")
ax.set_ylim(0, revenue_by_region.max() * 1.18)
fig.tight_layout()
plt.show()
plt.close(fig)


## 3. Line charts show ordered change

Sort time before plotting. Lines imply continuity or ordered progression, so do not connect unrelated categories as though one flows into the next.


### Plot a monthly time series

The missing values remain visible as gaps rather than being silently replaced.


In [ ]:
monthly_sorted = monthly.sort_values("month")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(
    monthly_sorted["month"],
    monthly_sorted["sales"],
    color="#7c3aed",
    linewidth=2,
    marker="o",
    markersize=3,
)
ax.set_title("Monthly sales over time")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (£000)")
ax.grid(axis="y", alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
plt.close(fig)


### Add a rolling mean

A six-month rolling mean reduces short-term noise. It is a descriptive smoother, not a forecast.


In [ ]:
monthly_sorted["sales_rolling_6"] = (
    monthly_sorted["sales"]
    .rolling(window=6, min_periods=3)
    .mean()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly_sorted["month"], monthly_sorted["sales"], label="Monthly", alpha=0.45)
ax.plot(monthly_sorted["month"], monthly_sorted["sales_rolling_6"], label="6-month mean", linewidth=2.5)
ax.set_title("Monthly sales with a rolling mean")
ax.set_ylabel("Sales (£000)")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
plt.close(fig)


## 4. Histograms show distributions

A histogram divides a numerical scale into intervals called bins. Bar area represents frequency. Different bin choices can reveal or hide structure, so state the choice and compare sensible alternatives.


### Plot transaction revenue

The distribution is right-skewed, so mean and median may differ.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sales["net_revenue"], bins=12, color="#d97706", edgecolor="white")
ax.axvline(sales["net_revenue"].median(), color="#7f1d1d", linestyle="--", label="Median")
ax.set_title("Distribution of transaction revenue")
ax.set_xlabel("Net revenue per transaction (£)")
ax.set_ylabel("Number of transactions")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)


### Compare two bin choices

The underlying observations are identical. Only the visual grouping changes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].hist(sales["net_revenue"], bins=6, color="#0891b2", edgecolor="white")
axes[0].set_title("6 bins")
axes[1].hist(sales["net_revenue"], bins=24, color="#0891b2", edgecolor="white")
axes[1].set_title("24 bins")
for axis in axes:
    axis.set_xlabel("Net revenue (£)")
axes[0].set_ylabel("Transactions")
fig.suptitle("Bin choice changes the visible pattern")
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Build for the audience

Every chart should have a question-led title, labelled axes with units, readable text, sufficient colour contrast, and a source or scope note when presented outside the notebook. Avoid decorative 3D effects and colour combinations that are difficult to distinguish.


### Create a concise executive chart

The title states the finding, while the subtitle information is placed in a small note.


In [ ]:
monthly_revenue = (
    sales.assign(month=sales["order_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month")["net_revenue"]
    .sum()
    .sort_index()
)
best_month = monthly_revenue.idxmax()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly_revenue.index, monthly_revenue.values, color="#1d4ed8", linewidth=2)
ax.scatter([best_month], [monthly_revenue.loc[best_month]], color="#dc2626", zorder=3)
ax.annotate(
    f"Peak £{monthly_revenue.loc[best_month]:,.0f}",
    xy=(best_month, monthly_revenue.loc[best_month]),
    xytext=(8, 10),
    textcoords="offset points",
)
ax.set_title("Monthly net revenue peaked in " + best_month.strftime("%B %Y"))
ax.set_ylabel("Net revenue (£)")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
plt.close(fig)


## Student coding lab

For each task, create the aggregation separately and inspect it before plotting.


        ### Student task 1: Bar chart of channel revenue

        <div class="task"><strong>Your job:</strong> Aggregate net revenue by channel, sort descending, and draw a labelled bar chart beginning at zero.</div>

        **Check your work**

        - The aggregation uses net_revenue.
- Bars are sorted intentionally.
- Title, axes, unit, and zero baseline are present.


In [ ]:
# STUDENT TASK 1
channel_revenue = pd.Series(dtype=float)
fig, ax = plt.subplots(figsize=(7, 4))
# Add the bar chart and labels.
plt.close(fig)
print(channel_revenue)


        ### Student task 2: Count categories honestly

        <div class="task"><strong>Your job:</strong> Create a horizontal bar chart of transaction count by product category with direct count labels.</div>

        **Check your work**

        - Counts come from rows rather than revenue.
- Horizontal order is readable.
- Direct labels match the counts.


In [ ]:
# STUDENT TASK 2
category_counts = sales["category"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
# Draw horizontal bars and add labels.
plt.close(fig)
print(category_counts)


        ### Student task 3: Create a chronological line chart

        <div class="task"><strong>Your job:</strong> Plot monthly orders from monthly_sales.csv. Sort dates, mark observations, label units, and add a light horizontal grid.</div>

        **Check your work**

        - Month uses datetime values.
- Rows are sorted before drawing.
- The line communicates ordered change.


In [ ]:
# STUDENT TASK 3
task_monthly = monthly.copy()
fig, ax = plt.subplots(figsize=(9, 4))
# Add the ordered line chart.
plt.close(fig)
print(task_monthly[["month", "orders"]].head())


        ### Student task 4: Compare raw and rolling values

        <div class="task"><strong>Your job:</strong> Add a 3-month rolling mean to monthly orders and plot raw and smoothed series with a legend.</div>

        **Check your work**

        - The rolling window is three observations.
- Both lines use the same axes.
- The smoothed line is described as a summary, not a forecast.


In [ ]:
# STUDENT TASK 4
rolling_data = monthly.sort_values("month").copy()
rolling_data["orders_rolling_3"] = np.nan
fig, ax = plt.subplots(figsize=(9, 4))
# Calculate and plot both series.
plt.close(fig)
print(rolling_data.head())


        ### Student task 5: Explain a histogram choice

        <div class="task"><strong>Your job:</strong> Draw satisfaction histograms using 5 and 15 bins. Add a comment explaining which representation is more meaningful for a 1-to-5 rating.</div>

        **Check your work**

        - The same data is used in both panels.
- Axes make bin choices clear.
- The comment recognises that satisfaction is a discrete rating.



**Optional extension:** Use exact integer-centred bins so each rating has its own bar.


In [ ]:
# STUDENT TASK 5
satisfaction = sales["satisfaction"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
# Add both histograms and labels.
plt.close(fig)
# Interpretation:


        ## Knowledge check

        Answer these without running new code first.

        1. Which question is a bar chart designed to answer?
2. Why sort a time series before plotting?
3. What does a rolling mean hide and reveal?
4. How can histogram bins change interpretation?
5. Why should most bar-chart value axes start at zero?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 adds scatter, stacked, and box plots, then combines them into a small business dashboard.
